In [ ]:
import glob
import platform

import pandas as pd

try:
    import torch
except ModuleNotFoundError:
    import sys, subprocess, importlib
    print("PyTorch not installed. Installing via pip (this may take several minutes)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchvision", "torchaudio"])
    importlib.invalidate_caches()
    import torch

import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [5]:
# Check the operating system - file location is different if its Windows or OS
if platform.system() == 'Windows':
    path = "G:/My Drive/EarthEngineData"
else:
    # For Holden's Mac
    path = "/Users/holden/Personal Projects/flame-flame-fruit/FireData"

files = glob.glob(path + "/*.csv")

In [ ]:
# Throw all the files into one large pandas dataframe (this took me 22m to run btw)
df_list = []
for file in files:
    # Best practice is probably to put this all in a try catch but it worked for me for now...
    df = pd.read_csv(file)
    
    # Only add some of the days with no fire, since with too many it will skew predictions (since fire is rare)
    no_fires = df[df['T21_max'] == 0].sample(frac=0.0013413685916579098, random_state=42) #choosing same ratio as RF model
    # Every day with fire
    fires = df[df['T21_max'] > 0]
    
    both = pd.concat([fires, no_fires])
    df_list.append(both)

#Combine all the dataframes into one
final_df = pd.concat(df_list, ignore_index=True) #ignore_index to reset the row numbers on each list
print("final dataset size: ", final_df.shape)

In [2]:
# Parse grid cell x,y indices out of system:index (format: YYYYMMDD_x,y)
# These represent which 4x4km grid cell in Colorado the row belongs to
final_df[['grid_x', 'grid_y']] = final_df['system:index'].str.split('_').str[1].str.split(',', expand=True).astype(int)

# Extracts the month number from the date (1-12) and adds it as a new column
final_df['month'] = pd.to_datetime(final_df['date']).dt.month

# Extracts year from the date and adds it as a new column
final_df['year'] = pd.to_datetime(final_df['date']).dt.year

# Creates a 0/1 column in case there is a fire
final_df['fire'] = (final_df['T21_max'] > 0).astype(int)

NameError: name 'final_df' is not defined

In [ ]:
# This just helps visualize the data frame's structure
print(final_df.columns.tolist())
final_df.head(5)

In [ ]:
# Split the data into training and testing sets
train_df = final_df[final_df['year'] < 2021]  # train on data before 2021
test_df = final_df[final_df['year'] >= 2021]  # test on data from 2021 and after

# Separate them into inputs and outputs
col_drop = ['fire', 'system:index', 'date', '.geo', 'T21_max', 'T21_mean', 'T21_stdDev']
X_train = train_df.drop(columns=col_drop)
X_test = test_df.drop(columns=col_drop)
y_train = train_df['fire']
y_test = test_df['fire']

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("Fire rate in train:", y_train.mean().round(3))
print("Fire rate in test: ", y_test.mean().round(3))

In [ ]:
# Standardize features - neural networks train much better with normalized inputs
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # use the same scaler fitted on training data

In [ ]:
# Convert numpy arrays to PyTorch tensors
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).to(device)

# Wrap in a DataLoader for batched training
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

In [ ]:
# Define a simple feedforward neural network for binary classification
class FireNet(nn.Module):
    def __init__(self, input_dim):
        super(FireNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()  # outputs a probability between 0 and 1
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

input_dim = X_train_scaled.shape[1]
model = FireNet(input_dim).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Class weight to address imbalance: fire days are rare so we penalize missing them more
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
print(f"pos_weight (penalty for missing a fire): {pos_weight.item():.2f}x")

# BCEWithLogitsLoss is numerically more stable, but we use BCE here since we already have Sigmoid
# Using pos_weight mirrors RandomForest's class_weight='balanced'
criterion = nn.BCELoss()

# Adam optimizer - adaptive learning rates per parameter, generally works well out of the box
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
# Training loop
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        preds = model(X_batch)

        # Apply per-sample weighting to handle class imbalance
        weights = torch.where(y_batch == 1, pos_weight.squeeze(), torch.ones(1).to(device))
        loss = (nn.BCELoss(reduction='none')(preds, y_batch) * weights).mean()

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {avg_loss:.4f}")

print("Training complete.")

In [ ]:
# Evaluate on the test set
model.eval()
with torch.no_grad():
    probs = model(X_test_tensor).cpu().numpy()

threshold = 0.6  # same threshold as the Random Forest model
predictions = (probs >= threshold).astype(int)

print(classification_report(y_test.values, predictions))